## Load Dependencies

In [ ]:
import torch
import torch.nn as nn
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import cv2
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split

## Load Data

In [ ]:
NUMBER_OF_PICTURES = 1818 # Total number of images in the dataset
SIZE=200 # Change image size if needed
DATASET = 'dataset' # Change this to the path of dataset image folder if needed
DATASET_EXCEL = 'dataset.xlsx' # Change this to the path of dataset excel file if needed

In [ ]:
def img_to_array(img):
    return np.asarray(img)

clean_data=[]
path = DATASET
files=os.listdir(path)
for i in tqdm(files):
    img=cv2.imread(path+'/'+i)  #Change 0 to 1 for color images
    # print(path+'/'+i)
    if(img.shape[0] > 0):
        img=cv2.resize(img,(SIZE, SIZE))
        img=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        clean_data.append(img_to_array(img))
    else:
        print(img.shape)
        print(i)
        print(path+'/'+i)


In [ ]:
clean_data=clean_data[:NUMBER_OF_PICTURES]
plt.imshow(clean_data[0],cmap='binary')
new_clean=[]
for i in clean_data:
    i=i/255
    new_clean.append(i)

In [ ]:
for i in new_clean:
    x=i
    x[x<0.5]=0

In [ ]:
plt.imshow(new_clean[0],cmap='binary')

## Data analysis - Removing the noise

In [ ]:
arr_ind =  np.arange(NUMBER_OF_PICTURES)
mean_row=[]
for i in arr_ind:
    mean_row.append(np.mean(new_clean[i][120]))
mean_array=np.array(mean_row)

## Check if most of the information are inside this range 

In [ ]:
plt.scatter(arr_ind,mean_array)
plt.plot(arr_ind,mean_array)
plt.show()

## Cropping the images - only keeping the sky portion

In [ ]:
for i in range(len(new_clean)):
    new_clean[i]=new_clean[i][:120]

In [ ]:
df=pd.read_excel(DATASET_EXCEL, engine='openpyxl').head(NUMBER_OF_PICTURES)

In [ ]:
# print(df)
df.isnull().any()

In [ ]:
y=df['PM2.5']
y=np.asarray(y).astype('float32')
print(len(y))
print(type(y))
print(y)

## Data Distribution

In [ ]:
y = df['PM2.5']
y = np.array(y)


check_0_50 = 0 
check_50_100 = 0
check_100_150 = 0
check_150_200 = 0
check_200_250 = 0
check_250_300 = 0
check_300_more = 0  

j = 0
for i in y:
    j = j + 1
    if i < 50:
        check_0_50 = check_0_50 + 1
    elif i >= 50 and i<100:
        # print(j-1)
        check_50_100 = check_50_100 + 1
    elif i >= 100 and i<150:
        # print(j-1)
        check_100_150 = check_100_150 + 1
    elif i >= 150 and i<200:
        # print(j-1)
        check_150_200 = check_150_200 + 1
    elif i >= 200 and i<250:
        # print(j-1)
        check_250_300 = check_250_300 + 1
    elif i >= 250 and i<300:
        # print(j-1)
        check_50_100 = check_50_100 + 1
    elif i >= 300:
        # print(j-1)
        check_300_more = check_300_more + 1

arr = [check_0_50, check_50_100, check_100_150, check_150_200, check_200_250, check_250_300, check_300_more]
z = [0, 50, 100, 150, 200, 250, 300]

# plt.style.use('ggplot')
# plt.bar(x=np.arange(y.size),height=y)
plt.hist(y, color=(0.2, 0.4, 0.6, 0.6),  edgecolor='blue')
plt.xlabel('PM2.5 Index')
plt.ylabel('Number of Images')
plt.gcf().set_dpi(300)
# plt.grid()
plt.show()

In [ ]:
new_clean[0].shape

In [ ]:
X=new_clean
print(len(X))
X=np.asarray(X).astype('float32')

print(type(X))

In [ ]:
plt.imshow(new_clean[0],cmap='binary')

## Dataset Splitting

In [ ]:
TEST_SIZE = 0.10 # Change test size if needed
SEED = 42 # Change random seed if needed

x_train, x_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size = TEST_SIZE, random_state = SEED,shuffle=True)

In [ ]:
x_train = x_train.reshape(-1,120,200, 3)

x_test = x_test.reshape(-1,120,200, 3)

In [ ]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

## Model Development

In [ ]:
import math
import time
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Keras defaults that differ from PyTorch defaults
KERAS_BN = dict(eps=1e-3, momentum=0.01)  # BatchNormalization(epsilon=1e-3, momentum=0.99)

def keras_init(m):
    """Initialise like Keras: glorot_uniform kernels, zero biases."""
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class KerasAdam(torch.optim.Optimizer):
    """Adam with the exact Keras update rule (epsilon=1e-7, added before bias correction,
    unlike torch.optim.Adam which adds it after)."""
    def __init__(self, params, lr=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-7):
        super().__init__(params, dict(lr=lr, beta_1=beta_1, beta_2=beta_2, epsilon=epsilon))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                if not state:
                    state['step'] = 0
                    state['m'] = torch.zeros_like(p)
                    state['v'] = torch.zeros_like(p)
                state['step'] += 1
                t = state['step']
                alpha = group['lr'] * math.sqrt(1 - group['beta_2'] ** t) / (1 - group['beta_1'] ** t)
                state['m'].lerp_(p.grad, 1 - group['beta_1'])
                state['v'].lerp_(p.grad.square(), 1 - group['beta_2'])
                p.addcdiv_(state['m'], state['v'].sqrt().add_(group['epsilon']), value=-alpha)

metrics_names = ['loss', 'mean_squared_error', 'root_mean_squared_error']

def to_tensor(x):
    """(N, H, W, C) numpy images -> (N, C, H, W) float32 tensor."""
    return torch.from_numpy(np.ascontiguousarray(np.asarray(x, dtype=np.float32).transpose(0, 3, 1, 2)))

def to_targets(y):
    return torch.from_numpy(np.asarray(y, dtype=np.float32).reshape(-1, 1))

def run_epoch(model, loss_fn, x, y, batch_size, optimizer=None, shuffle=False):
    """One pass over (x, y); trains when an optimizer is given. Returns [loss, mse, rmse]."""
    n = len(x)
    order = torch.randperm(n) if shuffle else torch.arange(n)
    sum_loss = sum_se = 0.0
    model.train(optimizer is not None)
    with torch.set_grad_enabled(optimizer is not None):
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            xb, yb = x[idx].to(device), y[idx].to(device)
            out = model(xb)
            loss = loss_fn(out, yb)
            if optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            sum_loss += loss.item() * len(idx)
            sum_se += torch.sum((out.detach() - yb) ** 2).item()
    return [sum_loss / n, sum_se / n, math.sqrt(sum_se / n)]

class History:
    def __init__(self):
        self.history = {}

def fit(model, optimizer, loss_fn, x, y, batch_size=32, epochs=1, verbose=1, validation_split=0.0, shuffle=True):
    """Equivalent of keras Model.fit: validation_split holds out the last fraction of x/y before shuffling."""
    x, y = to_tensor(x), to_targets(y)
    x_val = y_val = None
    if validation_split:
        split_at = int(math.floor(len(x) * (1.0 - validation_split)))
        x, x_val = x[:split_at], x[split_at:]
        y, y_val = y[:split_at], y[split_at:]
    steps = math.ceil(len(x) / batch_size)
    hist = History()
    for epoch in range(epochs):
        epoch_start = time.time()
        logs = dict(zip(metrics_names, run_epoch(model, loss_fn, x, y, batch_size, optimizer, shuffle)))
        if x_val is not None:
            val_scores = run_epoch(model, loss_fn, x_val, y_val, batch_size)
            logs.update({'val_' + k: v for k, v in zip(metrics_names, val_scores)})
        for k, v in logs.items():
            hist.history.setdefault(k, []).append(v)
        if verbose:
            print(f'Epoch {epoch + 1}/{epochs}')
            print(f'{steps}/{steps} - {time.time() - epoch_start:.0f}s - '
                  + ' - '.join(f'{k}: {v:.4f}' for k, v in logs.items()))
    return hist

def evaluate(model, loss_fn, x, y, batch_size=32, verbose=1):
    """Equivalent of keras Model.evaluate. Returns [loss, mse, rmse]."""
    scores = run_epoch(model, loss_fn, to_tensor(x), to_targets(y), batch_size)
    if verbose:
        print(f'{math.ceil(len(x) / batch_size)}/{math.ceil(len(x) / batch_size)} - '
              + ' - '.join(f'{k}: {v:.4f}' for k, v in zip(metrics_names, scores)))
    return scores

def predict(model, x, batch_size=32):
    """Equivalent of keras Model.predict. Returns a (N, 1) numpy array."""
    x = to_tensor(x)
    model.eval()
    outs = []
    with torch.no_grad():
        for start in range(0, len(x), batch_size):
            outs.append(model(x[start:start + batch_size].to(device)).cpu())
    return torch.cat(outs).numpy()

def summary(model, input_size=(3, 120, 200)):
    """Keras-style model.summary(): output shape and parameter count of every layer."""
    rows, hooks = [], []
    for name, module in model.named_modules():
        if len(list(module.children())) == 0:
            hooks.append(module.register_forward_hook(
                lambda m, i, o, name=name: rows.append(
                    (f'{name} ({m.__class__.__name__})', str((None,) + tuple(o.shape[1:])),
                     sum(p.numel() for p in m.parameters())))))
    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(torch.zeros(1, *input_size, device=next(model.parameters()).device))
    model.train(was_training)
    for h in hooks:
        h.remove()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Model: "{model.__class__.__name__}"')
    print('_' * 90)
    print(f'{"Layer (type)":<45}{"Output Shape":<30}{"Param #":>15}')
    print('=' * 90)
    for layer, shape, n in rows:
        print(f'{layer:<45}{shape:<30}{n:>15,}')
    print('=' * 90)
    print(f'Total params: {total:,}')
    print(f'Trainable params: {trainable:,}')
    print(f'Non-trainable params: {total - trainable:,}')
    print('_' * 90)

In [ ]:
model = nn.Sequential(
    nn.ZeroPad2d(padding=(3, 3, 3, 3)), # input_shape=(120,200,3), fed as (3,120,200)
    # First Conv2d Block
    nn.Conv2d(3, 32, (3,3)), nn.ReLU(),
    nn.BatchNorm2d(32, **KERAS_BN),
    # Second Conv2d Block
    nn.Conv2d(32, 64, (3,3)), nn.ReLU(),
    nn.BatchNorm2d(64, **KERAS_BN),
    nn.MaxPool2d(2,2),
    # Third Conv2d Block
    nn.Conv2d(64, 128, (3,3)), nn.ReLU(),
    nn.BatchNorm2d(128, **KERAS_BN),
    nn.MaxPool2d(2,2),
    # Fourth Conv2d Block
    nn.Conv2d(128, 256, (3,3)), nn.ReLU(),
    nn.BatchNorm2d(256, **KERAS_BN),
    nn.MaxPool2d(2,2),
    # Fifth and Final Conv2d Block
    nn.Conv2d(256, 512, (3,3)), nn.ReLU(),
    nn.BatchNorm2d(512, **KERAS_BN),
    nn.MaxPool2d(2,2),
    # Fully Connected Layer
    nn.Flatten(),
    nn.Linear(512*5*10, 128), nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(128, 1)  # activation='linear'
)
model.apply(keras_init)
model = model.to(device)


## Model Summary

In [ ]:
summary(model)

## Visualize the Model

In [ ]:
# visualkeras (used for the layered view in the Keras version) only supports Keras models.
# The PyTorch model's layers and output shapes are listed by summary(model) above.

## Images with Unique Values

In [ ]:
y = df['PM2.5']
y = np.array(y)

unique, counts = np.unique(y, return_counts=True)

xpoints = np.array(unique)
ypoints = np.array(counts)

plt.hist(xpoints)
plt.xlabel('PM2.5 Index')
plt.ylabel('Frequency of Images')
plt.gcf().set_dpi(300)
plt.show()

## Model Compilation

In [ ]:
# from torch.optim import RMSprop

LR = 0.000001 # Change learning rate if needed

optimum=KerasAdam(model.parameters(), lr=LR)
loss_fn = nn.L1Loss() # loss='mean_absolute_error'; metrics mean_squared_error and root_mean_squared_error are computed by fit/evaluate

## Fitting The Model

In [ ]:
import time

BATCH_SIZE = 8 # Change batch size if needed
EPOCHS = 350 # Change number of epochs if needed
VALIDATION_SPLIT = 0.1 # Change validation split if needed

start = time.process_time()
hist = fit(model, optimum, loss_fn, x_train, y_train,
           batch_size=BATCH_SIZE, epochs=EPOCHS, validation_split=VALIDATION_SPLIT)
print(time.process_time() - start)

## Print Predicted values

In [ ]:
predictions = predict(model, x_test)
pred=[]
for i in predictions:
    print(int(i[0]))
    pred.append(int(i[0]))

## Print True Values

In [ ]:
for i in y_test:
    print(i)

## Mean Difference from the true value

In [ ]:
diff=[]
for i in range(len(pred)):
    diff.append(abs(y_test[i]-pred[i]))
print(sum(diff) / len(diff))

## Print Differences

In [ ]:
for i in diff:
    print(i)

### Test Values

In [ ]:
eva = evaluate(model, loss_fn, x_test, y_test)
print(eva)

### MSE Train vs Validation Curve

In [ ]:
sns.set()

plt.plot(hist.history['mean_squared_error'], label='Train')
plt.plot(hist.history['val_mean_squared_error'], label='Validation')
plt.legend()
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


### MAE Train vs Validation Curve

In [ ]:
plt.plot(hist.history['loss'], label='Train')
plt.plot(hist.history['val_loss'], label='Validation')
plt.legend()
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


### RMSE Train vs Validation Curve

In [ ]:
plt.plot(hist.history['root_mean_squared_error'], label='Train')
plt.plot(hist.history['val_root_mean_squared_error'], label='Validation')
plt.legend()
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


In [ ]:
print(len(x_test)) # Test data size

### Near Values

In [ ]:
check_20 = 0 #check how many differences are less or equal 20
check_30 = 0 #check how many differences are less or equal 30
check_40 = 0 #check how many differences are less or equal 40
check_50 = 0 #check how many differences are less or equal 60
check_10 = 0 #check how many differences are less 10
check_50_more = 0 #check how many differences are more than 50
check_0 = 0
check_below_50 = 0
total=0
# for i in diff:
#     if i == 0:
#         check_0 = check_0 + 1
#         total+=1
#     elif i <= 10:
#         check_10 = check_10 + 1
#         total+=1
#     elif i <= 20:
#         total+=1
#         check_20 = check_20 + 1
#     elif i <= 30:
#         total+=1
#         check_30 = check_30 + 1
#     elif i <= 40:
#         total+=1
#         check_40 = check_40 + 1
#     elif i <= 50:
#         total+=1
#         check_50 = check_50 + 1
#     elif i > 50:
#         check_50_more = check_50_more + 1
#         # print("Too far")
j = 0
for i in diff:
    j = j + 1
    if i < 50:
        total+=1
        check_below_50 = check_below_50 + 1
    elif i >= 50:
        # print(j-1)
        check_50_more = check_50_more + 1
        # print("Too far")


# print(check_0)
# print(check_10)
# print(check_20)
print("# of pictures of AQI Difference of less than 50: ", check_below_50)
print("# of pictures of AQI Difference of 50 or more: ", check_50_more)
# print('Near:',total)

In [ ]:
near_percentage = (total/len(x_test) )* 100 # This percent of images can be considered as truly close to actual value since 50 difference is.

In [ ]:
print(near_percentage)

### Test Dataset Amount

In [ ]:
len(x_test)

In [ ]:
predic=[]
obser=[]
for i in range(len(pred)):
    predic.append(pred[i])
    obser.append(y_test[i])

In [ ]:
xpoints = np.array(predic)
ypoints = np.array(obser)

plt.scatter(xpoints, ypoints)
plt.xlabel('Predicted Values')
plt.ylabel('Observed Values')
plt.gcf().set_dpi(300)
plt.show()


In [ ]:
my_rho = np.corrcoef(predic, obser)

print(my_rho)

In [ ]:
# obser and predic must have identical shape
losas = nn.functional.l1_loss(torch.tensor(np.array(predic), dtype=torch.float32),
                              torch.tensor(np.array(obser), dtype=torch.float32)).numpy()  # scalar MAE

In [ ]:
losas

## $R^2$ Score

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(obser, predic)

In [ ]:
print(r2)

In [ ]:
print(x_test)

## Test Images with observed and predicted values

In [ ]:
from PIL import Image as im

val = 0

data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 3
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 4
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 5
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 6
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 7
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 10
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 11
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 12
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 14
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 15
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 16
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 17
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 18
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 19
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 20
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 21
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 72
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 172
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 181
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
# torch.save(model.state_dict(), "aqi_model.pt")   # Recommended: weights only (state_dict)
# torch.save(model, "aqi_model.pth")   # Whole pickled model